# Activity 3 — Exceptions and Assertions
**Module:** Advanced Programming — Week 2  
**University of York, MSc Computer Science**

---
## Exercise 1: Fixing MissingExceptions.ipynb

Each broken code block is reproduced with the identified error type(s) explained and corrected.

### Block 1 — Division and Modulo

**Errors generated from testing:**

| Test input | Error |
|---|---|
| `30, 3` | No error |
| `5, 45` | No error (correct branch runs) |
| `0, 0` | `ZeroDivisionError` — modulo by zero |
| `[Enter]` (empty) | `ValueError` — `int('')` fails |

In [ ]:
# ORIGINAL (broken)
# num1 = int(input("Enter a number:"))
# num2 = int(input("Enter another number:"))
# if num1 > num2:
#   if num1%num2 == 0:
#     print(num1, " is a multiple of ", num2)
# else:
#   if num2%num1 == 0:
#     print(num2, " is a multiple of ", num1)

# CORRECTED
try:
    num1 = int(input("Enter a number: "))
    num2 = int(input("Enter another number: "))

    if num1 > num2:
        if num2 == 0:
            raise ZeroDivisionError("Second number is zero — cannot compute modulo.")
        if num1 % num2 == 0:
            print(num1, "is a multiple of", num2)
        else:
            print(num1, "is NOT a multiple of", num2)
    else:
        if num1 == 0:
            raise ZeroDivisionError("First number is zero — cannot compute modulo.")
        if num2 % num1 == 0:
            print(num2, "is a multiple of", num1)
        else:
            print(num2, "is NOT a multiple of", num1)

except ValueError:
    print("Error: Please enter whole numbers only (no letters or empty input).")
except ZeroDivisionError as e:
    print(f"Error: {e}")

### Block 2 — List Index Out of Range

**Error generated:**  
`IndexError: list index out of range` — index 1000 is requested but the list will almost always have fewer than 1001 elements.  

The handler reports the actual list length and prints the last available element as a useful fallback.

In [ ]:
# ORIGINAL (broken)
# names = input("List of names:")
# nameList = names.split()
# print(nameList[1000])

# CORRECTED
try:
    names = input("List of names (space-separated): ")
    nameList = names.split()
    print(nameList[1000])

except IndexError:
    print(f"Error: The list contains {len(nameList)} name(s) — index 1000 does not exist.")
    if nameList:
        print(f"Last name in the list: '{nameList[-1]}'")

### Block 3 — Invalid Element in List

**Error generated:**  
`ValueError: invalid literal for int() with base 10: '3O'`  
The element `'3O'` contains the letter **O** (not the digit **0**) — a very easy data entry mistake.

**Key decision:** the `try/except` is placed *inside* the loop so valid elements are still processed after the bad one is skipped.

In [ ]:
# ORIGINAL (broken)
# randomList = ['67', 53, '3O', 72, '10']
# for i in randomList:
#   print(int(i) * 10)

# CORRECTED — try/except inside the loop so processing continues after the bad element
randomList = ['67', 53, '3O', 72, '10']

for i in randomList:
    try:
        print(int(i) * 10)
    except ValueError:
        print(f"Error: '{i}' cannot be converted to an integer "
              f"(check for letter/digit confusion, e.g. 'O' vs '0').")

# Expected output:
# 670
# 530
# Error: '3O' cannot be converted...
# 720
# 100

### Block 4 — File Input

**Errors that can occur:**

| Scenario | Exception |
|---|---|
| File does not exist | `FileNotFoundError` |
| No read permission | `PermissionError` |
| Path is a directory | `IsADirectoryError` |
| Other OS problem | `OSError` (base class — fallback) |

The `with` statement is also added so the file is always closed, even if an error occurs mid-read.

In [ ]:
# ORIGINAL (broken)
# fileName = input("Enter File:")
# print(open(fileName).read())

# CORRECTED
try:
    fileName = input("Enter file name: ")
    with open(fileName, 'r', encoding='utf-8') as f:
        print(f.read())

except FileNotFoundError:
    print(f"Error: '{fileName}' was not found. Check the file name and path.")
except PermissionError:
    print(f"Error: No permission to read '{fileName}'.")
except IsADirectoryError:
    print(f"Error: '{fileName}' is a directory, not a file.")
except OSError as e:
    # Fallback for any other OS-level file error
    print(f"Unexpected file error: {e}")

---
## Exercise 2: Assertions in Practice

**Assertions vs Exceptions — key distinction:**
- `assert` → detects **programmer errors** (wrong type passed to a function, impossible state)
- `try/except` → handles **runtime errors** (user input, missing files, network)

Assertions should never be used to validate user input — they can be disabled with `python -O`.

Both examples are drawn from my Ground2Tech engineering tools.

In [ ]:
# --- Example 1: Cost Deviation Calculator ---
# Railway Cost Deviation Tracker (Streamlit / FastAPI)

def calculate_deviation(planned: float, actual: float) -> float:
    """
    Calculate cost deviation as a decimal fraction.
    Positive = over budget. Negative = under budget.
    """
    # Preconditions — catch programmer errors before the function does any work
    assert isinstance(planned, (int, float)), \
        f"'planned' must be numeric, got {type(planned).__name__}"
    assert isinstance(actual, (int, float)), \
        f"'actual' must be numeric, got {type(actual).__name__}"
    assert planned > 0, \
        f"'planned' budget must be positive, got {planned}"
    assert actual >= 0, \
        f"'actual' cost cannot be negative, got {actual}"

    result = (actual - planned) / planned

    # Postcondition — flag absurd results that indicate a data error
    assert -1.0 <= result <= 10.0, \
        f"Deviation {result:.1%} is outside plausible range — check inputs"

    return result


# Valid calls
print(f"{calculate_deviation(5_500_000, 6_198_500):.2%}")  # 12.70% over budget
print(f"{calculate_deviation(1_000_000, 950_000):.2%}")    # -5.00% under budget

# The following would raise AssertionError during development:
# calculate_deviation(0, 5000)        # planned must be positive
# calculate_deviation('5M', 6000000)  # planned must be numeric

In [ ]:
# --- Example 2: CSV Row Parser ---
# Data ingestion layer for the cost tracker dashboard

def parse_project_row(row: dict) -> dict:
    """
    Parse and validate one row from a project CSV export.
    Expected keys: project_id, budget, actual_cost
    """
    # Precondition: the dict must have all required fields
    required_keys = {"project_id", "budget", "actual_cost"}
    assert required_keys.issubset(row.keys()), \
        f"Missing required fields: {required_keys - row.keys()}"

    project_id = row["project_id"]
    budget = float(row["budget"])
    actual = float(row["actual_cost"])
    deviation = calculate_deviation(budget, actual)

    result = {
        "project_id": project_id,
        "budget": budget,
        "actual_cost": actual,
        "deviation": deviation
    }

    # Postcondition: output has the expected structure
    assert set(result.keys()) == {"project_id", "budget", "actual_cost", "deviation"}, \
        "Output is missing expected keys — logic error in function"

    return result


# Test
test_row = {"project_id": "NR-2026-001", "budget": "1500000", "actual_cost": "1620000"}
print(parse_project_row(test_row))

# This would raise AssertionError — missing 'actual_cost':
# parse_project_row({"project_id": "NR-2026-001", "budget": "1500000"})

---
## Exercise 3: Discussion

### Q1. How was the exception handling broken down?

**Low-level (specific) handling was used throughout**, with one deliberate fallback.

Each block catches only the exception types that can realistically occur for that specific operation:
- Block 1 separately handles `ValueError` (bad input) and `ZeroDivisionError` (valid input, impossible operation) — the causes and fixes differ completely.
- Block 4 handles three distinct `OSError` subclasses individually, then uses `OSError` as a catch-all for anything unforeseen.
- Block 3 places the handler *inside the loop* — a structural decision, not just a type decision — so that one bad element does not abort the entire iteration.

A single broad `except Exception` would silence all errors but give the user no actionable information. The extra lines of code are worth it.

**Rule of thumb:** be as specific as possible at the top of the `except` chain, add a broad fallback at the bottom only if needed.

---

### Q2. Were there too few or too many assertions?

**Balance: one assertion per precondition, one per postcondition — no intermediate assertions.**

Asserting every intermediate value produces noise and does not improve confidence — if the inputs are correct and the algorithm is right, intermediate values follow automatically. The valuable checkpoints are:
- **Before** the function does work: verify the inputs are usable
- **After** the function finishes: verify the output is within expected bounds

In `calculate_deviation`, the postcondition `assert -1.0 <= result <= 10.0` caught a real bug during development: a test case where `planned` and `actual` had been accidentally swapped, producing a result of `-0.99` — technically valid but nonsensical. The assertion flagged it immediately.

**Final principle:** assertions are removed from production with `python -O`. If the check must always run (for user safety), it belongs in exception handling, not an assertion.